In [3]:
 #Install Required Dependencies
!pip install -q sentence-transformers umap-learn hdbscan bertopic datasets

In [4]:
#Step-by-Step Clustering Pipeline (UMAP + HDBSCAN)
#We will take a sample dataset, embed it, reduce its dimensionality, and run density-based clustering.
import umap
import hdbscan
import pandas as pd
from sentence_transformers import SentenceTransformer

# 1. Sample text documents
documents = [
    "The central processing unit CPU performs instructions.",
    "Graphics Processing Units GPU accelerate machine learning.",
    "Python features dynamic typing and readable syntax.",
    "JavaScript runs inside modern web browser engines.",
    "The football team won the championship match last night.",
    "Basketball players scored historical points in the finals."
]

# 2. Embed documents
embedder = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = embedder.encode(documents)

# 3. Reduce dimensionality using UMAP
umap_model = umap.UMAP(n_neighbors=2, n_components=2, min_dist=0.0, metric='cosine', random_state=42)
reduced_embeddings = umap_model.fit_transform(embeddings)

# 4. Cluster using HDBSCAN
cluster_model = hdbscan.HDBSCAN(min_cluster_size=2, metric='euclidean')
cluster_labels = cluster_model.fit_predict(reduced_embeddings)

df = pd.DataFrame({'Document': documents, 'Cluster': cluster_labels})
print("--- CLUSTERING RESULTS ---")
print(df)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

--- CLUSTERING RESULTS ---
                                            Document  Cluster
0  The central processing unit CPU performs instr...       -1
1  Graphics Processing Units GPU accelerate machi...       -1
2  Python features dynamic typing and readable sy...       -1
3  JavaScript runs inside modern web browser engi...       -1
4  The football team won the championship match l...       -1
5  Basketball players scored historical points in...       -1


/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


In [7]:
#Modular Topic Modeling with BERTopic
#Next, we leverage BERTopic to automate the end-to-end pipeline and extract topic keywords using class-based TF-IDF (c-TF-IDF).
from bertopic import BERTopic
import umap
import hdbscan # Import hdbscan

# Sample dataset of short tech/science texts
corpus = [
    "Quantum computing leverages superposition and entanglement.",
    "Quantum bits or qubits replace classical binary bits.",
    "Deep neural networks learn hierarchical feature representations.",
    "Transformer language models rely heavily on self-attention mechanisms.",
    "Astronomers observe distant galaxy clusters using space telescopes.",
    "Exoplanet discoveries provide insights into solar system formation."
]

# Initialize UMAP with parameters suitable for small datasets
# n_neighbors must be < len(corpus)
umap_model = umap.UMAP(n_neighbors=3, n_components=2, min_dist=0.0, metric='cosine', random_state=42)

# Initialize HDBSCAN with parameters suitable for small datasets
# min_cluster_size must be <= len(corpus)
hdbscan_model = hdbscan.HDBSCAN(min_cluster_size=2, metric='euclidean', prediction_data=True)

# Initialize and fit BERTopic with the custom UMAP and HDBSCAN models
topic_model = BERTopic(embedding_model="all-MiniLM-L6-v2", umap_model=umap_model, hdbscan_model=hdbscan_model)
topics, probs = topic_model.fit_transform(corpus)

# Display extracted topic info
topic_info = topic_model.get_topic_info()
print("-- TOPIC SUMMARY TABLE --")
print(topic_info[['Topic', 'Count', 'Name']])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

-- TOPIC SUMMARY TABLE --
   Topic  Count                           Name
0     -1      6  -1_bits_quantum_attention_and


## Generating Topic Labels with a Generative LLM

To complete Chapter 5, we add a text generation module (using Hugging Face or Cohere/OpenAI) to generate cohesive, natural language topic names from the extracted keywords.

In [19]:
from bertopic.representation import TextGeneration

# 1. Specify the model name
# Using 'distilgpt2' for compatibility with TextGeneration's default text-generation task
model_name = "distilgpt2"

# 2. Wrap the model name in BERTopic's TextGeneration representation model
# TextGeneration expects a model name (string) or a pipeline object.
representation_model = TextGeneration(model_name)

# 3. Pass the representation module to BERTopic
# We also pass the previously defined umap_model and hdbscan_model
# to ensure compatibility with the small dataset.
topic_model_llm = BERTopic(
    embedding_model="all-MiniLM-L6-v2",
    umap_model=umap_model, # Use the already defined umap_model
    hdbscan_model=hdbscan_model, # Use the already defined hdbscan_model
    representation_model=representation_model
)

topics, probs = topic_model_llm.fit_transform(corpus)

# Print custom generated topic titles
found_topics = False
for topic_id in set(topics):
    if topic_id != -1: # Ignore outliers (documents not assigned to a cluster)
       print(f"Topic {topic_id} Label:", topic_model_llm.get_topic(topic_id))
       found_topics = True

if not found_topics:
    print("No specific topics (other than outlier topic -1) were found given the small corpus.")

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


No specific topics (other than outlier topic -1) were found given the small corpus.
